# Enrichissement des métadonnées Turath-Art

Mémoire Omar Zeroual — préparation des données pour le chapitre 4B / discussion.

**Notebook autonome, séparé du pipeline d'évaluation VLM (v4).** Exécution → Tout exécuter.

Objectif : construire une table de métadonnées enrichies par artiste (bio, pays, médium) à partir de trois sources, par ordre de fiabilité :
1. **Kamel Lazaar Foundation** — collection publique, structurée, feu vert institutionnel explicite pour la recherche.
2. **Wikidata** — biographies courtes, 56.5% de couverture déjà validée.
3. **Mathqaf (Instagram)** — autorisation explicite de la fondatrice (Wadha), légendes précises avec noms d'artistes.

Sortie : `enriched_artist_metadata.csv`, une ligne par artiste des 425 de Turath-Art, avec les champs disponibles de chaque source. Ce fichier servira ensuite à construire des prompts enrichis pour une évaluation VLM comparative (baseline nom seul vs prompt enrichi), à faire dans un notebook séparé réutilisant le pipeline v4 déjà validé.

In [ ]:
!pip install -q pandas requests beautifulsoup4 instaloader gdown

## 1. Récupération de la liste des 425 artistes (indexation légère, pas de GPU)

In [ ]:
import gdown, zipfile, os, re
from pathlib import Path
import pandas as pd

DATA_DIR = "/content/turath_art"
os.makedirs(DATA_DIR, exist_ok=True)

FILE_ID = "1Gu87NPFgrtAi1qPP-8rgS0MPw1kTZ5fX"
zip_path = "/content/turath_art.zip"

if not os.path.exists(zip_path):
    gdown.download(id=FILE_ID, output=zip_path, quiet=False)

if not os.listdir(DATA_DIR):
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)

def clean_artist_name(raw_name):
    name = raw_name.replace("_", " ").replace("-", " ")
    name = re.sub(r"\bart\b\s*$", "", name, flags=re.IGNORECASE).strip()
    return name.title()

root = Path(DATA_DIR)
artist_folders = set()
for p in root.rglob("*"):
    if p.is_dir() and any(f.suffix.lower() in [".jpg", ".jpeg", ".png"] for f in p.glob("*.*")):
        artist_folders.add(p.name)

artist_list = sorted(clean_artist_name(a) for a in artist_folders)
print(f"{len(artist_list)} artistes détectés")
print(artist_list[:10])

425 artistes détectés
['Abdalla Omari', 'Abdallah Akar', 'Abdallah Benanteur', 'Abdallah Murad', 'Abdel Hadi El Gazzar', 'Abdel Kader Guermaz', 'Abdel Qader Hassan', 'Abdelkader Benchamma', 'Abdelkebir Rabi', 'Abderrahim Iqbi']


## 2. Kamel Lazaar Foundation — scraping de la collection publique

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

KLF_HEADERS = {"User-Agent": "TurathArtThesis-Research/1.0 (contact: omarzeroualpro@gmail.com)"}

def fetch_klf_collection_page(page_num):
    url = f"https://www.kamellazaarfoundation.org/collection?page={page_num}"
    try:
        r = requests.get(url, headers=KLF_HEADERS, timeout=15)
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Échec page {page_num}: {e}")
        return None

# ATTENTION : la structure HTML exacte (classes CSS, sélecteurs) doit être vérifiée
# en inspectant une page de la collection dans le navigateur avant de lancer ceci
# à grande échelle. Ce premier essai récupère la page 0 pour inspection.
test_soup = fetch_klf_collection_page(0)
if test_soup:
    print(test_soup.prettify()[:3000])

<!DOCTYPE html>
<html dir="ltr" lang="en" prefix="og: http://ogp.me/ns# content: http://purl.org/rss/1.0/modules/content/ dc: http://purl.org/dc/terms/ foaf: http://xmlns.com/foaf/0.1/ rdfs: http://www.w3.org/2000/01/rdf-schema# sioc: http://rdfs.org/sioc/ns# sioct: http://rdfs.org/sioc/types# skos: http://www.w3.org/2004/02/skos/core# xsd: http://www.w3.org/2001/XMLSchema#">
 <head>
  <link href="http://www.w3.org/1999/xhtml/vocab" rel="profile"/>
  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type">
   <link href="https://www.kamellazaarfoundation.org/favicon.ico" rel="shortcut icon" type="image/vnd.microsoft.icon"/>
   <meta content="Drupal 7 (https://www.drupal.org)" name="generator"/>
   <link href="https://www.kamellazaarfoundation.org/collection" rel="canonical"/>
   <link href="https://www.kamellazaarfoundation.org/collection" rel="shortlink"/>
   <meta content="Kamel Lazaar Foundation" pr

**Étape manuelle nécessaire ici** : le extrait HTML ci-dessus doit être inspecté pour identifier les bons sélecteurs (classes CSS ou balises) contenant le nom d'artiste, le titre, l'année, le pays. Une fois identifiés, adapte la fonction `parse_klf_artwork` ci-dessous avant de lancer le scraping complet. Ceci évite de scraper à l'aveugle avec des sélecteurs incorrects.

In [ ]:
def parse_klf_artwork(soup):
    """À adapter selon la structure HTML réelle observée à l'étape précédente."""
    artworks = []
    # Exemple générique à ajuster : chercher les liens vers des pages d'œuvres
    for link in soup.find_all("a", href=True):
        href = link["href"]
        if "/collection/" in href and href.count("/") > 2:
            text = link.get_text(strip=True)
            if text:
                artworks.append({"url": href, "text": text})
    return artworks

klf_records = []
MAX_PAGES = 100  # ajuster selon la taille réelle de la collection (~1300 œuvres)

for page_num in range(MAX_PAGES):
    soup = fetch_klf_collection_page(page_num)
    if soup is None:
        break
    artworks = parse_klf_artwork(soup)
    if not artworks:
        print(f"Plus de résultats à la page {page_num}, arrêt.")
        break
    klf_records.extend(artworks)
    time.sleep(1)

klf_df = pd.DataFrame(klf_records)
print(f"{len(klf_df)} entrées brutes récupérées depuis KLF")
klf_df.head(10)

Plus de résultats à la page 9, arrêt.
105 entrées brutes récupérées depuis KLF


,url,text
0,https://www.kamellazaarfoundation.org/collecti...,Dora Dalila CHEFFI
1,https://www.kamellazaarfoundation.org/collecti...,Dora Dalila CHEFFI
2,https://www.kamellazaarfoundation.org/collecti...,Atef Maatallah
3,https://www.kamellazaarfoundation.org/collecti...,Atef Maatallah
4,https://www.kamellazaarfoundation.org/collecti...,Atef Maatallah
5,https://www.kamellazaarfoundation.org/collecti...,Lawrence Abu Hamdan
6,https://www.kamellazaarfoundation.org/collecti...,Jordan Nassar
7,https://www.kamellazaarfoundation.org/collecti...,Jordan Nassar
8,https://www.kamellazaarfoundation.org/collecti...,Ali Tnani
9,https://www.kamellazaarfoundation.org/collecti...,Waqas Khan


In [ ]:
# Matching des entrées KLF avec les 425 artistes Turath-Art
artist_names_lower = {a.lower(): a for a in artist_list}

def find_artist_mentions(text):
    text_lower = text.lower()
    return [orig for low, orig in artist_names_lower.items() if low in text_lower]

if len(klf_df) > 0:
    klf_df["matched_artists"] = klf_df["text"].apply(find_artist_mentions)
    klf_matched = klf_df[klf_df["matched_artists"].apply(len) > 0]
    print(f"{klf_matched['matched_artists'].explode().nunique()} artistes Turath-Art trouvés dans KLF")
    klf_matched.head(10)

15 artistes Turath-Art trouvés dans KLF


## 3. Wikidata — biographies courtes (déjà validé)

In [ ]:
WD_HEADERS = {"User-Agent": "TurathArtThesis-Research/1.0 (contact: omarzeroualpro@gmail.com)"}

def search_wikidata_artist(name):
    try:
        r = requests.get("https://www.wikidata.org/w/api.php", params={
            "action": "wbsearchentities", "search": name, "language": "en",
            "format": "json", "type": "item", "limit": 3
        }, headers=WD_HEADERS, timeout=10)
        r.raise_for_status()
        return r.json().get("search", [])
    except Exception as e:
        print(f"Échec pour '{name}': {e}")
        return []

wikidata_matches = []
for artist in artist_list:
    results_wd = search_wikidata_artist(artist)
    hit = None
    for r in results_wd:
        desc = r.get("description", "").lower()
        if any(kw in desc for kw in ["painter", "artist", "sculptor", "photographer", "calligrapher"]):
            hit = r
            break
    wikidata_matches.append({"artist": artist, "found": hit is not None,
                              "wikidata_id": hit["id"] if hit else None,
                              "wikidata_description": hit.get("description") if hit else None})
    time.sleep(0.2)

wikidata_df = pd.DataFrame(wikidata_matches)
coverage = wikidata_df["found"].mean()
print(f"Couverture Wikidata : {wikidata_df['found'].sum()}/{len(wikidata_df)} artistes ({coverage:.1%})")
wikidata_df[wikidata_df["found"]].head(10)

Couverture Wikidata : 240/425 artistes (56.5%)


,artist,found,wikidata_id,wikidata_description
0,Abdalla Omari,True,Q55235519,Syrian painter and filmmaker
2,Abdallah Benanteur,True,Q2820914,Algerian painter (1931-2017)
7,Abdelkader Benchamma,True,Q2821080,French artist
10,Abdul Hay Mosallam Zarara,True,Q8761616,artist
13,Abdul Qadir Al Rassam,True,Q4665592,Iraqi painter (1882-1952)
17,Abdul Rahman Mowakket,True,Q2885140,Syrian sculptor
19,Abdulhalim Radwi,True,Q140249223,Saudi painter and sculptor (1939-2006)
20,Abdullah Al Muharraqi,True,Q94629321,Bahraini painter
22,Abdulnasser Gharem,True,Q19282148,Saudi Arabian artist
23,Achraf Touloub,True,Q55265156,"Moroccan visual artist, b. 1986"


## 4. Mathqaf (Instagram) — légendes avec accord explicite de la fondatrice

In [ ]:
import instaloader

L = instaloader.Instaloader(
    download_pictures=False, download_videos=False, download_video_thumbnails=False,
    download_geotags=False, download_comments=False, save_metadata=False, compress_json=False
)

from getpass import getpass
L.login("eau.mare", getpass("Mot de passe Instagram :  "))

profile = instaloader.Profile.from_username(L.context, "mathqaf")

mathqaf_posts = []
count = 0
MAX_POSTS = 500

for post in profile.get_posts():
    caption = post.caption or ""
    matched_artists = find_artist_mentions(caption)
    if matched_artists:
        mathqaf_posts.append({
            "post_url": f"https://www.instagram.com/p/{post.shortcode}/",
            "date": str(post.date),
            "caption": caption,
            "hashtags": list(post.caption_hashtags),
            "matched_artists": matched_artists,
        })
    count += 1
    if count >= MAX_POSTS:
        break
    time.sleep(1.5)

mathqaf_df = pd.DataFrame(mathqaf_posts)
print(f"{len(mathqaf_df)} posts avec mention d'un artiste Turath-Art sur {count} scannés")
mathqaf_df.head(10)

Mot de passe Instagram :  ··········


LoginException: Login: Checkpoint required. Point your browser to /auth_platform/?apc=AdpbKOVrJkYhwsKbl-SoiBtyLcvQFWLZETD2JGwAw36_V-YhX2KI01Kv_ZnAx6Lh9nGxn6-bLf9wGednYafIgRkw5Etzl2bGkdt0ZdjCdw2MCJaIsWa9n_QiWXk4NIzY3IuRTYNkIVS3yHqGr1tcKhOQRV6PuOuypUkawvPibKx6IsiUj_BgZDU8jQv6KCmu1ql4EcJBFZkMz4ttmutk8Rjz9ZWxt1JDEBlKC_HwCSS-57SZfjPL76T5xot8BTDcww71y4pnSxMmtR9vsiZRpiO1MINLp1myEbtxR79dx_hFlSezNByIeTkoCmN4NyJuMh_zV8hsymzlmzeUb5Ve6XGyXo2QbWlXAuemBFdg0MZ6eLHjDKtmLOpKztlek7jNsZs6kh-zM7kHkfpIz-SWsLAo-UGTLDaSYTbxBRU0kfNz8h1d60reStInf9voA6oWcdZadLdBU_7dTdNvZ7RtsR2IMXZDQojgZZUxOIVGKPy2SRUANSGx0xkuN7-jKAReCzBOVQCr4J9PDcHR58w4EZegjx--wWYV5POPJtt7s3RzC0wttFwoXPXVOrb6VIoRQoDVu-CT2ubeT47X7eq_mOsK0h0nfs2I-xi2QYzn9p3Jpx8RoaQcUYGN5hnhC10qsLZmZhAN8oCnO54yQAoEHTh_xSQ_DA - follow the instructions, then retry.

## 5. Fusion des trois sources en une table de métadonnées par artiste

In [ ]:
enriched = pd.DataFrame({"artist": artist_list})

enriched = enriched.merge(wikidata_df[["artist", "wikidata_description"]], on="artist", how="left")

if len(klf_df) > 0 and "matched_artists" in klf_df.columns:
    klf_exploded = klf_df.explode("matched_artists").rename(columns={"matched_artists": "artist"})
    klf_agg = klf_exploded.groupby("artist")["text"].apply(lambda x: "; ".join(x.unique()[:3])).reset_index()
    klf_agg = klf_agg.rename(columns={"text": "klf_info"})
    enriched = enriched.merge(klf_agg, on="artist", how="left")
else:
    enriched["klf_info"] = None

if len(mathqaf_df) > 0:
    mq_exploded = mathqaf_df.explode("matched_artists").rename(columns={"matched_artists": "artist"})
    mq_agg = mq_exploded.groupby("artist")["caption"].apply(lambda x: " | ".join(x.unique()[:2])).reset_index()
    mq_agg = mq_agg.rename(columns={"caption": "mathqaf_caption"})
    enriched = enriched.merge(mq_agg, on="artist", how="left")
else:
    enriched["mathqaf_caption"] = None

n_any_source = enriched[["wikidata_description", "klf_info", "mathqaf_caption"]].notna().any(axis=1).sum()
print(f"Couverture combinée (au moins une source) : {n_any_source}/{len(enriched)} artistes ({n_any_source/len(enriched):.1%})")

enriched.to_csv("/content/enriched_artist_metadata.csv", index=False)
enriched.head(15)